# Entrenar EN AWS aunque la cuota de entrenamiento sea 0 — con un *Processing Job*

Este es el **plan B** de `linear-learner-houses-LOCAL.ipynb`. Misma meta, pero el entrenamiento
ocurre **dentro de AWS**, no en la laptop.

## El truco

Las cuotas de SageMaker no son una sola bolsa: cada **tipo de trabajo** tiene la suya.

| Cuota en **us-east-2** | Valor |
|---|---|
| `ml.m4.xlarge` **for training job usage** | 0 ❌ |
| *cualquier* instancia **for training job usage** | 0 ❌ |
| `ml.t3.medium` **for processing job usage** | **4** ✅ |
| `ml.t3.large` for processing job usage | 4 ✅ |
| `ml.m4.xlarge` **for endpoint usage** | **6** ✅ |

Un **Processing Job** es otro tipo de trabajo de SageMaker (pensado para ETL, feature engineering,
validación de datos) y **tiene su propia cuota**. Nada impide que el código que corre adentro sea
un entrenamiento: es un contenedor con tu script, tus datos montados de S3, y una carpeta de salida
que se sube sola a S3 al terminar.

Es decir: **cuota 0 de entrenamiento no significa que no puedas entrenar en AWS.**

## Diferencias contra un Training Job de verdad

Para que quede honesto con los alumnos, un Processing Job **no** es un Training Job:

| | Training Job | Processing Job (lo que hacemos aquí) |
|---|---|---|
| Cuota que consume | `... for training job usage` | `... for processing job usage` |
| Métricas en la consola | sí, con gráficas | no, solo logs de CloudWatch |
| Spot / checkpointing | sí | no |
| Hyperparameter Tuning Jobs | sí | no |
| Model Registry automático | sí | hay que registrarlo a mano |
| Genera `model.tar.gz` | automático | **lo generamos nosotros en el script** |

Sirve perfecto para enseñar el ciclo completo, y para entrenar de verdad cuando las cuotas no dan.

In [1]:
# === Compatibilidad del SDK de SageMaker (funciona en v2 y en v3) ===
import os
os.environ['SAGEMAKER_SUPPRESS_V2_WARNING'] = '1'

import sagemaker
from importlib.metadata import version

try:
    from sagemaker.session import Session, get_execution_role
    sagemaker.Session = Session
    sagemaker.get_execution_role = get_execution_role
    print("API clásica disponible")
except ImportError:
    print("Este SDK ya no expone la API clásica; instala 'sagemaker<3'")

print("SDK SageMaker:", version("sagemaker"))

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/kevininofuente/Library/Application Support/sagemaker/config.yaml


API clásica disponible
SDK SageMaker: 2.257.6


# Carregamento da base de dados

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import numpy as np

In [3]:
base_casas = pd.read_csv('house_prices.csv')
base_casas

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,20140521T000000,360000.0,3,2.50,1530,1131,3.0,0,0,...,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,6600060120,20150223T000000,400000.0,4,2.50,2310,5813,2.0,0,0,...,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,1523300141,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,0,...,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,291310100,20150116T000000,400000.0,3,2.50,1600,2388,2.0,0,0,...,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


> El análisis exploratorio (correlaciones, missing values, describe) está en
> `linear-learner-houses-LOCAL.ipynb`. Aquí vamos directo al punto: el preprocesamiento
> y el entrenamiento **ocurren dentro de AWS**, en `code/train_processing.py`.

# Configuraciones SageMaker

### La región importa

El bucket está en **us-east-2** y ahí están las cuotas. Si `boto3` toma otra región del
`~/.aws/config`, SageMaker falla diciendo que no puede leer el modelo de S3 y culpando al rol IAM
— cuando en realidad el rol está bien y lo que no coincide es la región. Así que la fijamos:

In [4]:
import boto3
from sagemaker.session import Session

REGION = 'us-east-2'
boto3.setup_default_session(region_name=REGION)

bucket = 'aws-sagemaker-datapath-ml-regresion-linear-learner-k'
subcarpeta_modelo = 'modelos/house-prices/linear-learner-processing'
subcarpeta_dataset = 'datasets/house-prices'

# default_bucket: para que el SDK suba el script y los inputs a NUESTRO bucket
session = Session(default_bucket=bucket)

print('Región:', boto3.Session().region_name)
print('Bucket:', bucket)

Región: us-east-1
Bucket: aws-sagemaker-datapath-ml-regresion-linear-learner-k


In [5]:
# Rol de ejecución: el Processing Job y el endpoint correrán con este rol
iam = boto3.client('iam')
roles = [r['Arn'] for pagina in iam.get_paginator('list_roles').paginate()
         for r in pagina['Roles']
         if r['RoleName'].startswith('AmazonSageMaker-ExecutionRole')]
role = roles[0]
print('Rol:', role)

Rol: arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20260818T191520


### Subimos el CSV crudo a S3

El Processing Job corre en una máquina de AWS: no ve el disco de la laptop. Todo lo que necesite
tiene que estar en S3. **Los datos viven en S3** — esa es la regla.

In [6]:
llave_csv = f'{subcarpeta_dataset}/raw/house_prices.csv'
boto3.resource('s3').Bucket(bucket).upload_file('house_prices.csv', llave_csv)

s3_csv = f's3://{bucket}/{llave_csv}'
print('CSV crudo en:', s3_csv)

CSV crudo en: s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/datasets/house-prices/raw/house_prices.csv


# Entrenamiento con un Processing Job

`SKLearnProcessor` levanta el contenedor oficial `sagemaker-scikit-learn` en una `ml.t3.medium`
y corre el script que le pasemos. Las rutas que monta dentro del contenedor son:

```
/opt/ml/processing/input/data   <- el CSV (lo declaramos como ProcessingInput)
/opt/ml/processing/input/lib    <- code/linear_learner_local.py (nuestro algoritmo)
/opt/ml/processing/output       <- todo lo que dejemos aquí se sube solo a S3
```

Fíjate en el segundo input: en vez de instalar nuestro algoritmo como dependencia, lo **montamos
como un archivo más**. El script hace `sys.path.insert(0, '/opt/ml/processing/input/lib')` y lo
importa. Es la forma más simple de llevar código propio a un contenedor ajeno.

El script (`code/train_processing.py`) hace exactamente lo del notebook original: corta las
columnas 3 a 18, separa 70/30 con `random_state=0`, entrena 16 modelos, elige el mejor, y deja
en la carpeta de salida el `model.tar.gz`, las métricas y el `test.csv` que apartó.

In [7]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

procesador = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type='ml.t3.medium',      # <- la cuota que SÍ tenemos (4)
    instance_count=1,
    base_job_name='entrena-linear-learner',
    sagemaker_session=session,
)

destino_salida = f's3://{bucket}/{subcarpeta_modelo}/output'
print('La salida del job irá a:', destino_salida)

La salida del job irá a: s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output


In [8]:
# Esta celda tarda ~5 minutos: AWS levanta la t3.medium, baja el contenedor y corre el script.
# Los logs que verás son la salida real del script dentro de AWS.
procesador.run(
    code='code/train_processing.py',
    inputs=[
        ProcessingInput(source=s3_csv,
                        destination='/opt/ml/processing/input/data',
                        input_name='datos'),
        ProcessingInput(source='code/linear_learner_local.py',
                        destination='/opt/ml/processing/input/lib',
                        input_name='algoritmo'),
    ],
    outputs=[
        ProcessingOutput(source='/opt/ml/processing/output',
                         destination=destino_salida,
                         output_name='modelo'),
    ],
    arguments=['--num-models', '16', '--epocas', '200'],
    wait=True,
    logs=True,
)

descripcion = procesador.latest_job.describe()
print('\nEstado:', descripcion['ProcessingJobStatus'])
print('Instancia:', descripcion['ProcessingResources']['ClusterConfig']['InstanceType'])

INFO:sagemaker:Creating processing-job with name entrena-linear-learner-2026-08-20-23-08-02-064


.

.

.

.

.

... (7 outputs left out)

.

.

Leyendo house_prices.csv
  21,613 filas x 21 columnas
  entrenamiento=(15129, 16)  test=(6484, 16)
Entrenando 16 modelos (Adam, mini-batches)...
  modelo  1/16  learning_rate=0.001  wd=0        -> MSE validacion (normalizado) = 0.362387
  modelo  2/16  learning_rate=0.001  wd=1e-06    -> MSE validacion (normalizado) = 0.361709
  modelo  3/16  learning_rate=0.001  wd=0.0001   -> MSE validacion (normalizado) = 0.363174
  modelo  4/16  learning_rate=0.001  wd=0.01     -> MSE validacion (normalizado) = 0.362321
  modelo  5/16  learning_rate=0.005  wd=0        -> MSE validacion (normalizado) = 0.363612
  modelo  6/16  learning_rate=0.005  wd=1e-06    -> MSE validacion (normalizado) = 0.364328
  modelo  7/16  learning_rate=0.005  wd=0.0001   -> MSE validacion (normalizado) = 0.363910
  modelo  8/16  learning_rate=0.005  wd=0.01     -> MSE validacion (normalizado) = 0.359240
  modelo  9/16  learning_rate=0.01   wd=0        -> MSE validacion (normalizado) = 0.361730
  modelo 10/16  learning_ra


Estado: Completed
Instancia: ml.t3.medium


In [9]:
# ¿Qué dejó el job en S3?
s3 = boto3.client('s3')
prefijo = f'{subcarpeta_modelo}/output/'
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefijo).get('Contents', []):
    print(f"{obj['Size']:>12,} bytes  s3://{bucket}/{obj['Key']}")

       1,516 bytes  s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output/metricas.json
       1,845 bytes  s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output/model.json
       1,108 bytes  s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output/model.tar.gz
     644,444 bytes  s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output/test.csv


### Las métricas que calculó AWS

El script dejó un `metricas.json` con el resultado en test y con la tabla completa de la
competencia de los 16 modelos (`num_models=16`).

In [10]:
import json, io

def leer_de_s3(llave):
    return s3.get_object(Bucket=bucket, Key=llave)['Body'].read()

metricas_job = json.loads(leer_de_s3(f'{prefijo}metricas.json'))

print('Métricas en test (calculadas dentro de AWS):')
for k, v in metricas_job['test'].items():
    print(f'   {k:<5} {v:>14,.4f}')

tabla = (pd.DataFrame(metricas_job['competencia'])
         .sort_values('mse_val')
         .reset_index(drop=True))
print(f'\nLos {len(tabla)} modelos que compitieron (num_models=16), del mejor al peor:')
display(tabla.head(8))

Métricas en test (calculadas dentro de AWS):
   RMSE    207,103.8070
   MAE     125,225.6751
   R2            0.6876



Los 16 modelos que compitieron (num_models=16), del mejor al peor:


,lr,wd,mse_val
0,0.005,0.010000,0.359240
1,0.010,0.000001,0.359815
2,0.001,0.000001,0.361709
3,0.010,0.000000,0.361730
4,0.001,0.010000,0.362321
5,0.001,0.000000,0.362387
6,0.001,0.000100,0.363174
7,0.005,0.000000,0.363612


# Deploy del endpoint

El artefacto ya está en S3, generado por AWS. El deploy es **idéntico** al del notebook local:
el endpoint no sabe ni le importa dónde se entrenó el modelo.

In [11]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.utils import name_from_base

s3_modelo = f's3://{bucket}/{prefijo}model.tar.gz'
nombre_endpoint = 'linear-learner-houses-aws'

modelo_sagemaker = SKLearnModel(
    model_data=s3_modelo,
    role=role,
    entry_point='inference.py',
    source_dir='code',
    framework_version='1.2-1',
    py_version='py3',
    sagemaker_session=session,
    code_location=f's3://{bucket}/{subcarpeta_modelo}',
    name=name_from_base('linear-learner-houses-aws'),
)
print('Artefacto:', s3_modelo)
print('Imagen:   ', modelo_sagemaker.serving_image_uri(boto3.Session().region_name, 'ml.m4.xlarge'))

Artefacto: s3://aws-sagemaker-datapath-ml-regresion-linear-learner-k/modelos/house-prices/linear-learner-processing/output/model.tar.gz
Imagen:    683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


> ⚠️ **Cuidado con la cuota y con el costo.** `ml.m4.xlarge for endpoint usage` es **4**. Si ya
> tienes vivo el endpoint del notebook local (`linear-learner-houses-local`), este sería un
> segundo endpoint, o sea una segunda instancia cobrándose (≈ US$0.28/h cada una).
>
> Para la clase, lo normal es tener **uno solo**: borra el otro antes, o sáltate esta celda y usa
> el endpoint que ya tienes.

In [ ]:
import time

cliente_sm = boto3.client('sagemaker')

# --- 1) el endpoint, si ya existía de una corrida anterior
try:
    cliente_sm.describe_endpoint(EndpointName=nombre_endpoint)
    print(f'"{nombre_endpoint}" ya existía -> lo borramos')
    cliente_sm.delete_endpoint(EndpointName=nombre_endpoint)
    while True:
        try:
            cliente_sm.describe_endpoint(EndpointName=nombre_endpoint); time.sleep(10)
        except cliente_sm.exceptions.ClientError:
            break
    print('   endpoint borrado')
except cliente_sm.exceptions.ClientError:
    print(f'El endpoint "{nombre_endpoint}" no existía')

# --- 2) el endpoint config: el SDK lo crea con el MISMO nombre que el endpoint.
#        Si no lo borras, la segunda corrida falla con
#        "Cannot create endpoint config ... name already exists"
try:
    cliente_sm.describe_endpoint_config(EndpointConfigName=nombre_endpoint)
    cliente_sm.delete_endpoint_config(EndpointConfigName=nombre_endpoint)
    print(f'   endpoint config "{nombre_endpoint}" borrado')
except cliente_sm.exceptions.ClientError:
    print(f'El endpoint config "{nombre_endpoint}" no existía')

predictor = modelo_sagemaker.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge',
    endpoint_name=nombre_endpoint,
)
print('\nEndpoint listo:', predictor.endpoint_name)

# Evaluación con el conjunto de prueba EXACTO que apartó AWS

Aquí está la ventaja de que el job haya guardado su `test.csv`: evaluamos el endpoint con
**las mismas filas** que el entrenamiento nunca vio, sin tener que reproducir el split en la
laptop y sin riesgo de mezclar datos.

In [13]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

# El test.csv que dejó el Processing Job (primera columna = precio real)
df_test = pd.read_csv(io.BytesIO(leer_de_s3(f'{prefijo}test.csv')))
y_test = df_test['price'].values
X_test = df_test.drop(columns=['price']).values

print(f'{len(df_test):,} filas de prueba, {X_test.shape[1]} features')
display(df_test.head(3))

6,484 filas de prueba, 16 features


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long
0,297000.0,2.0,1.50,1430.0,1650.0,3.0,0.0,0.0,3.0,7.0,1430.0,0.0,1999.0,0.0,98125.0,47.7222,-122.290
1,1578000.0,4.0,3.25,4670.0,51836.0,2.0,0.0,0.0,4.0,12.0,4670.0,0.0,1988.0,0.0,98005.0,47.6350,-122.164
2,562100.0,2.0,0.75,1440.0,3700.0,1.0,0.0,0.0,3.0,7.0,1200.0,240.0,1914.0,0.0,98107.0,47.6707,-122.364


In [14]:
# Mandamos el test completo al endpoint, en tandas
predicciones = []
TANDA = 500
for inicio in range(0, len(X_test), TANDA):
    lote = X_test[inicio:inicio + TANDA]
    csv = '\n'.join(','.join(map(str, fila)) for fila in lote)
    predicciones.extend(p['score'] for p in predictor.predict(csv)['predictions'])

predicciones = np.array(predicciones)

residuo = y_test - predicciones
r2 = 1 - float(np.sum(residuo ** 2)) / float(np.sum((y_test - y_test.mean()) ** 2))
print('Métricas del ENDPOINT:')
print(f'   RMSE  {np.sqrt(np.mean(residuo ** 2)):>14,.4f}')
print(f'   MAE   {np.mean(np.abs(residuo)):>14,.4f}')
print(f'   R2    {r2:>14,.4f}')
print('\nMétricas que reportó el Processing Job dentro de AWS:')
for k, v in metricas_job['test'].items():
    print(f'   {k:<5} {v:>14,.4f}')
print('\nSi coinciden, el modelo entrenado en AWS y el que sirve el endpoint son el mismo.')

Métricas del ENDPOINT:
   RMSE    207,103.8071
   MAE     125,225.6704
   R2            0.6876

Métricas que reportó el Processing Job dentro de AWS:
   RMSE    207,103.8070
   MAE     125,225.6751
   R2            0.6876

Si coinciden, el modelo entrenado en AWS y el que sirve el endpoint son el mismo.


### Invocación cruda con `boto3`

El mismo contrato que `linear-learner`: entra `text/csv`, sale `{"predictions":[{"score":…}]}`.

In [15]:
runtime = boto3.client('sagemaker-runtime')

casa = np.array([3, 1.00, 1180, 5650, 1.0, 0, 0, 3, 7, 1180, 0, 1955, 0, 98178, 47.5112, -122.257])

respuesta = runtime.invoke_endpoint(
    EndpointName=nombre_endpoint,
    ContentType='text/csv',
    Body=','.join(map(str, casa)).encode('utf-8'),
)
cuerpo = json.loads(respuesta['Body'].read().decode('utf-8'))
print('JSON:', cuerpo)
print(f"Predicción: ${cuerpo['predictions'][0]['score']:,.2f}")
print(f'Precio real de esa casa: ${base_casas.iloc[0, 2]:,.2f}')

JSON: {'predictions': [{'score': 224453.1252949949}]}
Predicción: $224,453.13
Precio real de esa casa: $221,900.00


# Apagar el endpoint

Se cobra por hora encendida. Descomenta cuando termines.

In [16]:
# predictor.delete_endpoint()
# predictor.delete_model()

for e in boto3.client('sagemaker').list_endpoints()['Endpoints']:
    print(e['EndpointName'], '|', e['EndpointStatus'], '|', e['CreationTime'])

linear-learner-houses-aws | InService | 2026-08-20 20:13:38.014000-03:00
linear-learner-houses-local | InService | 2026-08-20 20:10:05.393000-03:00
